# ❤️ Heart Disease Classification
**Model:** LightGBM + SHAP explainability  
**Dataset:** UCI Heart Disease (Cleveland)  
**Target:** Binary — presence of heart disease  

**Techniques:**
- `class_weight='balanced'` to handle class imbalance
- Stratified K-Fold cross-validation
- SHAP waterfall plots per prediction
- Threshold tuning for recall vs precision tradeoff

In [ ]:
!git clone https://github.com/Aurovindhya/ml-portfolio-suite.git
%cd ml-portfolio-suite
!pip install -q lightgbm scikit-learn pandas numpy matplotlib seaborn shap

In [ ]:
# Load UCI Heart Disease from sklearn or fetch directly
import pandas as pd
import numpy as np
import os

try:
    from sklearn.datasets import fetch_openml
    data = fetch_openml('heart-disease', version=1, as_frame=True)
    df = data.frame
    df.columns = df.columns.str.lower()
    df['target'] = (df['num'] > 0).astype(int)
    print('OpenML heart disease dataset loaded:', df.shape)
except Exception:
    # UCI Cleveland fallback
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
    cols = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','target']
    df = pd.read_csv(url, header=None, names=cols, na_values='?')
    df['target'] = (df['target'] > 0).astype(int)
    print('UCI Cleveland dataset loaded:', df.shape)

os.makedirs('data', exist_ok=True)
df.to_csv('data/heart_disease.csv', index=False)
print(f'Class balance: {df.target.value_counts().to_dict()}')
df.head()

In [ ]:
# EDA
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df.groupby('target')['age'].hist(ax=axes[0], bins=20, alpha=0.7, label=['No disease','Disease'])
axes[0].set_title('Age by Disease Status'); axes[0].legend()

sns.boxplot(data=df, x='target', y='thalach', ax=axes[1])
axes[1].set_title('Max Heart Rate by Disease'); axes[1].set_xticklabels(['No','Yes'])

corr = df[['age','trestbps','chol','thalach','oldpeak','target']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[2])
axes[2].set_title('Feature Correlation')
plt.tight_layout(); plt.show()

In [ ]:
import sys; sys.path.insert(0, '.')
from modules.classification.heart_disease.model import train
metrics = train('data/heart_disease.csv')
print('Heart disease model metrics:', metrics)

In [ ]:
# Threshold tuning
import pickle, numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
from modules.classification.heart_disease.model import FEATURE_COLS

with open('weights/heart_disease.pkl', 'rb') as f:
    bundle = pickle.load(f)

X = df[[c for c in FEATURE_COLS if c in df.columns]].fillna(df.median())
y = df['target']
probs = bundle['model'].predict_proba(X)[:, 1]

thresholds = np.arange(0.2, 0.8, 0.05)
precisions = [precision_score(y, probs >= t, zero_division=0) for t in thresholds]
recalls    = [recall_score(y, probs >= t, zero_division=0) for t in thresholds]
f1s        = [f1_score(y, probs >= t, zero_division=0) for t in thresholds]

plt.figure(figsize=(10, 5))
plt.plot(thresholds, precisions, label='Precision', marker='o')
plt.plot(thresholds, recalls,    label='Recall',    marker='s')
plt.plot(thresholds, f1s,        label='F1',        marker='^')
plt.axvline(0.5, color='grey', ls='--', label='Default threshold')
plt.xlabel('Threshold'); plt.ylabel('Score')
plt.title('Threshold Tuning — Heart Disease Classifier')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

In [ ]:
# SHAP waterfall for a single patient
import shap

explainer = shap.TreeExplainer(bundle['model'])
shap_vals  = explainer.shap_values(X)
sv_class1  = shap_vals[1] if isinstance(shap_vals, list) else shap_vals

shap.summary_plot(sv_class1, X, plot_type='bar', show=True)

# Waterfall for patient 0
shap.waterfall_plot(
    shap.Explanation(values=sv_class1[0], base_values=explainer.expected_value[1], data=X.iloc[0]),
    show=True
)